# 22-тарау. Мектепке арналған қазақша чат-бот

*«Жасанды интеллект негіздері» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 22.1. Жобаның мақсаты

## 22.2. 1-қадам. Ниеттер мен мысалдар

In [ ]:
intents = {
    "сәлем": ["сәлем", "сәлеметсіз бе", "қайырлы таң", "армысыз", "сәлем бот",
              "қалың қалай", "қайырлы күн"],
    "қоштасу": ["сау бол", "көріскенше", "қош бол", "кейін сөйлесеміз",
                "сау болыңыз", "жақсы түн", "мен кеттім"],
    "рахмет": ["рахмет", "көп рахмет", "үлкен рахмет",
               "көмектескеніңе рахмет", "жарайсың", "керемет көмек",
               "алғыс айтамын"],
    "кесте": ["ертең қандай сабақтар бар", "сабақ кестесі қандай",
              "бүгін неше сабақ", "дүйсенбіде қандай пәндер",
              "сабақ қашан басталады", "кесте өзгерді ме",
              "бірінші сабақ қандай"],
    "үй_тапсырмасы": ["үй тапсырмасы қандай", "математикадан не берілді",
                      "үй жұмысы бар ма", "тапсырманы қашан тапсыру керек",
                      "физикадан не оқу керек", "үйге не берді",
                      "тапсырма көп пе"],
    "кітапхана": ["кітапхана қашан ашық", "кітапты қалай алуға болады",
                  "кітапхана қайда", "кітапты қайтару мерзімі",
                  "кітапханада оқулық бар ма", "кітап алғым келеді",
                  "кітапхана жұмыс уақыты"],
    "асхана": ["асханада не бар", "түскі ас қашан", "бүгінгі мәзір қандай",
               "асхана қайда орналасқан", "тамақ қанша тұрады",
               "тамақтанғым келеді", "асхана жұмыс істей ме"],
    "жи": ["жасанды интеллект деген не", "сен кімсің", "сен роботсың ба",
           "нейрондық желі қалай жұмыс істейді", "сені кім жасады",
           "сен ойлай аласың ба", "машиналық оқыту деген не"],
}

answers = {
    "сәлем": "Сәлем! Мен мектептің көмекші ботымын. Сабақ кестесі, "
             "үй тапсырмасы, кітапхана немесе асхана туралы сұрай аласың.",
    "қоштасу": "Сау бол! Сабағың сәтті өтсін!",
    "рахмет": "Оқасы жоқ! Тағы сұрағың болса, жаза бер.",
    "кесте": "Сабақтар сағат 8:30-да басталады. Толық кесте мектеп "
             "сайтында және бірінші қабаттағы тақтада.",
    "үй_тапсырмасы": "Үй тапсырмасын электрондық күнделіктен көре аласың. "
                     "Түсініксіз болса, пән мұғаліміне жаз.",
    "кітапхана": "Кітапхана дүйсенбі–жұма күндері 9:00–17:00 аралығында "
                 "ашық. Кітапты 14 күнге алуға болады.",
    "асхана": "Асхана екінші қабатта. Түскі ас үлкен үзілісте, 11:15-те.",
    "жи": "Мен — қарапайым ЖИ ботымын. Сұрағыңның мағынасын машиналық "
          "оқыту арқылы анықтаймын.",
}

texts, labels = [], []
for intent, examples in intents.items():
    texts += examples
    labels += [intent] * len(examples)
print("Мысалдар:", len(texts), " Ниеттер:", len(intents))

## 22.3. 2-қадам. TF-IDF: сөздердің маңыздылығы

## 22.4. 3-қадам. Жіктеуішті оқыту

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

classifier = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)),
    LogisticRegression(C=10, max_iter=1000),
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(classifier, texts, labels, cv=cv)
print(f"Кросс-валидация дәлдігі: {scores.mean():.0%}")

classifier.fit(texts, labels)

## 22.5. 4-қадам. Сенімділік шегі

In [ ]:
def reply(message, threshold=0.3):
    probs = classifier.predict_proba([message.lower()])[0]
    best = probs.argmax()
    intent, confidence = classifier.classes_[best], probs[best]
    if confidence < threshold:
        return "Кешір, сұрағыңды түсінбедім. Басқаша жазып көрші.", confidence
    return answers[intent], confidence

tests = ["Сәлем, қалайсың?", "ертеңгі сабақтар қандай?",
         "кітапханадан кітап алуға бола ма", "физикадан үйге не берілді",
         "футбол матчы қашан?", "сен шынымен ойлайсың ба?"]
for message in tests:
    answer, conf = reply(message)
    print(f"👤 {message}\n🤖 ({conf:.0%}) {answer}\n")

## 22.6. 5-қадам. Сөйлесу циклі

In [ ]:
print("🤖 Сәлем! Сұрағыңды жаз (шығу үшін: сау бол)")
while True:
    message = input("👤 ")
    answer, conf = reply(message)
    print("🤖", answer)
    if classifier.predict([message.lower()])[0] == "қоштасу":
        break